In [1]:
#cell 1: nạp manifest_v2.csv (chia theo phiên quay), dựng đặc trưng 126 cho train/val/test (CUDA)
import math, time, json, collections
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook không train trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
assert (BASE / "manifest_v2.csv").exists(), "Chưa có manifest_v2.csv, chạy 28_SplitV2.ipynb trước"
m = pd.read_csv(BASE / "manifest_v2.csv")
assert m.tap.isin(["train", "val", "test"]).all() and (m.groupby("nhom").tap.nunique() == 1).all(), "manifest_v2.csv sai: thiếu tap hoặc có nhóm nằm ở 2 tập"
Tmax, FPS_DATA = int(m.khung.max()), 120

def load_base_features(f):                                   # giống 19: cổ tay làm gốc, chia cỡ bàn tay, thêm vận tốc
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]
    lm = lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]
    pos = lm.reshape(len(lm), 63)
    has = ~np.isnan(pos[:, 0])
    vel = np.vstack([np.zeros((1, 63), np.float32), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32); Mk = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = load_base_features(f); X[i, :len(x)], Mk[i, :len(x)] = x, h
    return (torch.tensor(X, device=dev), torch.tensor(df.nhan.values, dtype=torch.long, device=dev), torch.tensor(Mk, device=dev))

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
assert [len(meta[t]) for t in ("train", "val", "test")] == [221, 45, 39], "Số video train/val/test không khớp manifest_v2 mong đợi (221/45/39)"
data = {t: build(meta[t]) for t in meta}
Xte, yte, Mte = data["test"]                                  # chỉ dùng ở cell 10 và 11; không dùng để chọn model

IS_LAB = (meta["test"].nguon == "dataset_labeled").to_numpy()   # test: person_01 (người khác) hoặc Sang
SUB = {"all": np.ones(len(IS_LAB), bool), "nguoi_khac": IS_LAB, "sang": ~IS_LAB}
SUB = {k: torch.tensor(np.where(v)[0], device=dev) for k, v in SUB.items()}

print("CUDA:", torch.cuda.get_device_name(0), "| Tmax:", Tmax)
for t, (X, y, Mk) in data.items():
    print(t, tuple(X.shape), "| video:", len(y), "| khung có tay:", int(Mk.sum().item()))
print(m.pivot_table(index="nguon", columns="tap", values="path", aggfunc="count")[["train", "val", "test"]])

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Tmax: 355
train (221, 355, 126) | video: 221 | khung có tay: 28205
val (45, 355, 126) | video: 45 | khung có tay: 4662
test (39, 355, 126) | video: 39 | khung có tay: 4867
tap              train   val  test
nguon                             
Dataset_bonus     66.0   NaN  24.0
Dataset_hand      63.0  18.0   NaN
dataset_labeled   92.0  27.0  15.0


In [2]:
#cell 2: đặc trưng 546, xoay 3D, 3 kiểu augmentation (04 "all", wide45_lownoise, 13 step2), TCN, hàm loss
PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)

def pos_vel(pos, hand_mask):                                 # (b,T,21,3) hoặc (b,T,63) -> 126 đặc trưng: tọa độ + vận tốc
    pos = pos.reshape(pos.shape[0], pos.shape[1], 63)
    vel = torch.zeros_like(pos); vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    return torch.cat([pos, vel], dim=-1)

def features_from_pos(pos, hand_mask):                       # giống 13/15: 126 gốc + 210 khoảng cách + 210 vận tốc khoảng cách
    pos = torch.where(hand_mask[:, :, None, None], pos, torch.zeros_like(pos))
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I, :] - pos[:, :, PAIR_J, :], dim=-1) * hand_mask[:, :, None]
    dist_vel = torch.zeros_like(dist); dist_vel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pair_valid
    return torch.cat([pos_vel(pos, hand_mask), dist, dist_vel], dim=-1)

def geo_from_base(X, hand_mask):
    return features_from_pos(X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3), hand_mask)

def rotate_deg(pos, ax, ay, az):                             # xoay 3D theo góc (độ) ax, ay, az, mỗi cái (b,1,1)
    ax, ay, az = [a * math.pi / 180 for a in (ax, ay, az)]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    return torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)

def augment_pos(X, hand_mask, rot_deg, noise_std):           # mỗi clip một góc xoay ngẫu nhiên + lật gương + nhiễu -> (b,T,21,3)
    b, T = X.shape[:2]
    pos = rotate_deg(X[..., :63].reshape(b, T, 21, 3).clone(), *[(torch.rand(b, 1, 1, device=X.device) * 2 - 1) * d for d in rot_deg])
    valid = hand_mask[:, :, None, None]
    flip = torch.rand(b, 1, 1, device=X.device) < 0.5
    pos[..., 0] = torch.where(flip, -pos[..., 0], pos[..., 0])
    pos = pos + torch.randn_like(pos) * noise_std * valid
    return torch.where(valid, pos, torch.zeros_like(pos))

def augment_v2(X, hm):    return pos_vel(augment_pos(X, hm, (10, 25, 20), 0.01), hm)             # 04_Trainv2 "all"
def augment_wide(X, hm):  return pos_vel(augment_pos(X, hm, (40, 45, 40), 0.005), hm)            # wide45_lownoise
def augment_step2(X, hm): return features_from_pos(augment_pos(X, hm, (40, 45, 40), 0.005), hm)  # 13_Task8_Step2

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.pad = 2 * d; self.conv = nn.Conv1d(c, c, 3, dilation=d); self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__(); self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)]); self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

def loss_acc(net, X, y, Mk):
    out = net(X)
    loss_each = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")
    correct = out.argmax(-1) == y[:, None]
    return (loss_each * Mk).sum() / Mk.sum().clamp(min=1), (correct & Mk).sum() / Mk.sum().clamp(min=1)

ID = lambda X, Mk: X
VARIANTS = {   # n_in: số đặc trưng | aug: augmentation lúc train | feat: đổi đặc trưng gốc 126 -> đầu vào model | cfg: dùng để nhận checkpoint cũ
    "tcn_v2":          dict(n_in=126, aug=augment_v2,    feat=ID,           cfg={"aug": "04_all", "n_in": 126}),
    "wide45_lownoise": dict(n_in=126, aug=augment_wide,  feat=ID,           cfg={"aug": "wide45_lownoise", "rot": (40, 45, 40), "noise": 0.005, "n_in": 126}),
    "step2":           dict(n_in=546, aug=augment_step2, feat=geo_from_base, cfg={"aug": "13_step2", "n_in": 546}),
}
Xv, yv, Mv = data["val"]
VAL = {k: v["feat"](Xv, Mv) for k, v in VARIANTS.items()}    # đặc trưng val dựng một lần
print("Đã định nghĩa:", list(VARIANTS), "| đầu vào:", {k: v["n_in"] for k, v in VARIANTS.items()})

NameError: name 'torch' is not defined

In [3]:
#cell 3: cấu hình train và hàm fit (train một model, dừng sớm theo val loss, lưu checkpoint) ở thư mục MỚI
EPOCHS, PATIENCE, BS, SEEDS = 120, 20, 16, [0, 1, 2, 3]
CK = BASE / "checkpoints_task9_v2"            # thư mục mới, KHÔNG ghi đè checkpoints_task9 (chứa model đang chạy game)
ck_path = lambda name, seed: CK / name / f"seed{seed}" / "TCN.pt"

def fit(name, seed, path, tr, va, tag, force=False):   # tr, va = (X, y, mask); tag ghi vào cấu hình để phân biệt thí nghiệm
    v = VARIANTS[name]; (Xt, yt, Mt), (Xv_, yv_, Mv_) = tr, va
    cfg = {**v["cfg"], "variant": name, "seed": seed, "epochs": EPOCHS, "patience": PATIENCE,
           "n_train": len(yt), "n_val": len(yv_), "tag": tag}                    # đổi dữ liệu/siêu tham số thì train lại
    if path.exists() and not force:
        ck = torch.load(path, map_location=dev, weights_only=False)
        if ck.get("cfg") == cfg:
            print(f"{tag} {name} seed {seed}: nạp checkpoint có sẵn | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return
        print(f"{tag} {name} seed {seed}: checkpoint có cấu hình khác, sẽ train lại.")
    VALF = v["feat"](Xv_, Mv_)
    torch.manual_seed(seed)
    net = TCN(v["n_in"]).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    best, wait, hist, t0 = float("inf"), 0, [], time.time()
    for ep in range(1, EPOCHS + 1):
        net.train(); perm = torch.randperm(len(Xt), device=dev); total = 0.0
        for s0 in range(0, len(Xt), BS):
            ids = perm[s0:s0 + BS]
            loss, _ = loss_acc(net, v["aug"](Xt[ids], Mt[ids]), yt[ids], Mt[ids])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            total += loss.item() * len(ids)
        net.eval()
        with torch.no_grad(): vl, va_ = loss_acc(net, VALF, yv_, Mv_)
        hist.append({"epoch": ep, "train_loss": total / len(Xt), "val_loss": float(vl.item()), "val_acc": float(va_.item())})
        if hist[-1]["val_loss"] < best:
            best, wait = hist[-1]["val_loss"], 0
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best, "hist": hist,
                        "model": "TCN", "variant": f"{tag}_{name}_seed{seed}", "cfg": cfg}, path)
        else:
            wait += 1
        if wait >= PATIENCE: break
    ck = torch.load(path, map_location=dev, weights_only=False)
    ck.update(stop_epoch=ep, seconds=time.time() - t0); torch.save(ck, path)
    print(f"{tag} {name} seed {seed}: tốt nhất epoch {ck['epoch']} | dừng ở epoch {ep} | val_loss {ck['val_loss']:.4f} | {ck['seconds']:.0f}s")
print("Checkpoint sẽ lưu ở:", CK)

Checkpoint sẽ lưu ở: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9_v2


In [4]:
#cell 4: train 12 model (3 họ x 4 seed) trên train mới; epoch tốt nhất chọn theo val loss; có checkpoint đúng cấu hình thì tự nạp
FORCE_RETRAIN = False   # True chỉ khi muốn train lại tất cả
for name in VARIANTS:
    for seed in SEEDS:
        fit(name, seed, ck_path(name, seed), data["train"], data["val"], "chinh", FORCE_RETRAIN)
    print("-" * 80)
print("Xong. Checkpoint ở:", CK)

chinh tcn_v2 seed 0: tốt nhất epoch 32 | dừng ở epoch 52 | val_loss 0.1249 | 6s
chinh tcn_v2 seed 1: tốt nhất epoch 27 | dừng ở epoch 47 | val_loss 0.1387 | 4s
chinh tcn_v2 seed 2: tốt nhất epoch 47 | dừng ở epoch 67 | val_loss 0.1176 | 5s
chinh tcn_v2 seed 3: tốt nhất epoch 55 | dừng ở epoch 75 | val_loss 0.1138 | 5s
--------------------------------------------------------------------------------
chinh wide45_lownoise seed 0: tốt nhất epoch 64 | dừng ở epoch 84 | val_loss 0.1180 | 7s
chinh wide45_lownoise seed 1: tốt nhất epoch 16 | dừng ở epoch 36 | val_loss 0.1507 | 3s
chinh wide45_lownoise seed 2: tốt nhất epoch 106 | dừng ở epoch 120 | val_loss 0.1086 | 9s
chinh wide45_lownoise seed 3: tốt nhất epoch 55 | dừng ở epoch 75 | val_loss 0.1243 | 6s
--------------------------------------------------------------------------------
chinh step2 seed 0: tốt nhất epoch 30 | dừng ở epoch 50 | val_loss 0.1230 | 4s
chinh step2 seed 1: tốt nhất epoch 93 | dừng ở epoch 113 | val_loss 0.1005 | 10s


In [5]:
#cell 5: bảng tóm tắt 12 model trên tập val (nạp lại từ checkpoint) và lưu train_summary.csv
rows = []
for name, v in VARIANTS.items():
    for seed in SEEDS:
        p = ck_path(name, seed)
        if not p.exists(): print("Thiếu checkpoint:", p); continue
        ck = torch.load(p, map_location=dev, weights_only=False)
        net = TCN(v["n_in"]).to(dev); net.load_state_dict(ck["w"]); net.eval()
        with torch.no_grad(): vl, va = loss_acc(net, VAL[name], yv, Mv)
        rows.append({"họ": name, "seed": seed, "epoch tốt nhất": ck["epoch"], "epoch dừng": ck.get("stop_epoch", np.nan),
                     "val_loss": vl.item(), "val_acc": va.item(), "giây": ck.get("seconds", np.nan)})
summary = pd.DataFrame(rows)
summary.to_csv(CK / "train_summary.csv", index=False)
display(summary.round(4))
display(summary.groupby("họ")[["val_loss", "val_acc"]].agg(["mean", "min", "max"]).round(4))
print("Đã lưu:", CK / "train_summary.csv")

,họ,seed,epoch tốt nhất,epoch dừng,val_loss,val_acc,giây
0,tcn_v2,0,32,52,0.1249,0.9575,5.8841
1,tcn_v2,1,27,47,0.1387,0.9547,3.5256
2,tcn_v2,2,47,67,0.1176,0.9610,4.8555
3,tcn_v2,3,55,75,0.1138,0.9687,5.4650
4,wide45_lownoise,0,64,84,0.1180,0.9558,6.8424
5,wide45_lownoise,1,16,36,0.1507,0.9483,2.7900
6,wide45_lownoise,2,106,120,0.1086,0.9635,9.1419
7,wide45_lownoise,3,55,75,0.1243,0.9603,5.6187
8,step2,0,30,50,0.1230,0.9520,4.4271
9,step2,1,93,113,0.1005,0.9541,10.0700


val_loss                 val_acc                
                    mean     min     max    mean     min     max
họ                                                              
step2             0.1127  0.1005  0.1230  0.9533  0.9520  0.9541
tcn_v2            0.1237  0.1138  0.1387  0.9605  0.9547  0.9687
wide45_lownoise   0.1254  0.1086  0.1507  0.9570  0.9483  0.9635

Đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9_v2\train_summary.csv


In [6]:
#cell 6: nạp 12 checkpoint, tạo 15 ứng viên (từng seed + ensemble 4 seed của mỗi họ), hàm tính xác suất cho một tập bất kỳ
NETS = {}
for fam, v in VARIANTS.items():
    for s in SEEDS:
        p = ck_path(fam, s); assert p.exists(), f"Thiếu {p}; chạy cell 4 trước"
        net = TCN(v["n_in"]).to(dev); net.load_state_dict(torch.load(p, map_location=dev, weights_only=False)["w"]); NETS[(fam, s)] = net.eval()

CANDS = {}
for fam in VARIANTS:
    CANDS.update({f"{fam} s{s}": [(fam, s)] for s in SEEDS})
    CANDS[f"{fam} ens{len(SEEDS)}"] = [(fam, s) for s in SEEDS]

@torch.no_grad()
def last_probs(ks, X, Mk):                                   # xác suất trung bình của các net trong ks, mọi khung: (b,T,3)
    inp = {f: VARIANTS[f]["feat"](X, Mk) for f in {k[0] for k in ks}}
    return torch.stack([NETS[k](inp[k[0]]).softmax(-1) for k in ks]).mean(0)

@torch.no_grad()
def all_probs(X, Mk):                                        # xác suất của mọi ứng viên trên tập (X, Mk); đặc trưng và net chỉ tính một lần
    inp = {f: v["feat"](X, Mk) for f, v in VARIANTS.items()}
    P1 = {k: n(inp[k[0]]).softmax(-1) for k, n in NETS.items()}
    return {name: torch.stack([P1[k] for k in ks]).mean(0) for name, ks in CANDS.items()}
print("Ứng viên:", len(CANDS), "|", list(CANDS))

Ứng viên: 15 | ['tcn_v2 s0', 'tcn_v2 s1', 'tcn_v2 s2', 'tcn_v2 s3', 'tcn_v2 ens4', 'wide45_lownoise s0', 'wide45_lownoise s1', 'wide45_lownoise s2', 'wide45_lownoise s3', 'wide45_lownoise ens4', 'step2 s0', 'step2 s1', 'step2 s2', 'step2 s3', 'step2 ens4']


In [7]:
#cell 7: chấm điểm 15 ứng viên trên VAL (không dùng test): góc xoay giả lập 0/15/30/45/60, tiêu chí Decision (ngưỡng 0.5 đến 0.9)
THRS = [0.5, 0.6, 0.7, 0.8, 0.9]

def metrics(P, y, Mk):
    conf, pred = P.max(-1)
    ok = (pred == y[:, None]) & Mk
    r = {"acc_all": ok.sum().item() / max(Mk.sum().item(), 1) * 100}
    for t in THRS:
        k = int(round(t * 100)); hit = ok & (conf >= t); has = hit.any(1)
        r[f"v{k}"] = has.float().mean().item() * 100                                           # % video có khung đúng và đủ tin cậy
        r[f"ms{k}"] = (hit.float().argmax(1).float()[has].mean().item() / FPS_DATA * 1000) if has.any() else float("nan")
    r["tb"] = float(np.mean([r[f"v{int(round(t * 100))}"] for t in THRS]))
    sel = (conf >= 0.7) & Mk
    r["accc70"] = (ok & sel).sum().item() / max(sel.sum().item(), 1) * 100                      # độ chính xác khi tin cậy >= 0.7
    r["cov70"] = sel.sum().item() / max(Mk.sum().item(), 1) * 100                               # độ phủ ở ngưỡng 0.7
    return r

def rotated(X, Mk, mag, g):                                  # xoay mỗi video một góc ngẫu nhiên trong ±mag độ (cả 3 trục)
    ang = lambda: (torch.rand(len(X), 1, 1, device=dev, generator=g) * 2 - 1) * mag
    pos = rotate_deg(X[..., :63].reshape(len(X), X.shape[1], 21, 3), ang(), ang(), ang())
    return pos_vel(torch.where(Mk[:, :, None, None], pos, torch.zeros_like(pos)), Mk)

MAGS, REPEAT = [0, 15, 30, 45, 60], 5
g = torch.Generator(device=dev).manual_seed(0)
rows = []
for mag in MAGS:
    acc = {}
    for _ in range(1 if mag == 0 else REPEAT):
        Xr = Xv if mag == 0 else rotated(Xv, Mv, mag, g)      # mọi ứng viên chấm trên cùng dữ liệu xoay
        for name, P in all_probs(Xr, Mv).items():
            acc.setdefault(name, []).append(metrics(P, yv, Mv))
    for name, v in acc.items():
        rows.append({"model": name, "goc": mag, **{k: float(np.nanmean([d[k] for d in v])) for k in v[0]}})
    print("xong góc", mag)
resv = pd.DataFrame(rows)
resv.to_csv(CK / "compare_val.csv", index=False)
print("Đã lưu:", CK / "compare_val.csv")

xong góc 0
xong góc 15
xong góc 30
xong góc 45
xong góc 60
Đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9_v2\compare_val.csv


In [8]:
#cell 8: bảng xếp hạng trên VAL, chọn model theo quy tắc: xếp theo TB mọi góc; chênh dưới TOL điểm thì lấy bản nhanh hơn
TOL = 2.0                                                    # điểm phần trăm (tham số tự đặt; val chỉ 45 clip, mỗi clip khoảng 2 điểm)
xw, mw = Xv[:1, :128], Mv[:1, :128]
def bench(ks, n=100):                                        # ms cho một lần dự đoán (dựng đặc trưng + model), cửa sổ 128 khung như webcam
    for _ in range(10): last_probs(ks, xw, mw)
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(n): last_probs(ks, xw, mw)
    torch.cuda.synchronize(); return (time.perf_counter() - t) / n * 1000

at0 = lambda col: resv[resv.goc == 0].set_index("model")[col]
tab = pd.DataFrame({
    "TB 0°": at0("tb"), "TB mọi góc": resv.groupby("model").tb.mean(),
    "acc 0°": at0("acc_all"), "acc 45°": resv[resv.goc == 45].set_index("model").acc_all,
    "acc≥0.7": at0("accc70"), "phủ 0.7": at0("cov70"), "ms đoán đúng 0.7": at0("ms70"),
    "ms/dự đoán": pd.Series({k: bench(ks) for k, ks in CANDS.items()}),
}).sort_values("TB mọi góc", ascending=False)
print("BẢNG XẾP HẠNG TRÊN VAL (45 clip):"); display(tab.round(1))
single = tab[~tab.index.str.contains("ens")]
print("Độ lệch giữa các seed của cùng một họ (TB mọi góc):")
display(single.groupby([n.rsplit(" ", 1)[0] for n in single.index])["TB mọi góc"].agg(["mean", "min", "max"]).round(1))

top = tab["TB mọi góc"].iloc[0]
near = tab[tab["TB mọi góc"] >= top - TOL]
BEST = near["ms/dự đoán"].idxmin()
print(f"Đứng đầu bảng: {tab.index[0]} ({top:.1f}). Trong vòng {TOL} điểm có {len(near)} ứng viên: {list(near.index)}")
print(f"MODEL ĐƯỢC CHỌN (nhanh nhất trong nhóm đó): {BEST}")

BẢNG XẾP HẠNG TRÊN VAL (45 clip):


,TB 0°,TB mọi góc,acc 0°,acc 45°,acc≥0.7,phủ 0.7,ms đoán đúng 0.7,ms/dự đoán
step2 s0,100.0,100.0,95.2,95.2,98.9,91.8,84.6,1.0
step2 s1,100.0,100.0,95.4,95.3,99.7,91.1,82.2,1.2
step2 s2,100.0,100.0,95.3,94.3,99.4,90.7,79.8,1.1
step2 ens4,100.0,99.9,96.7,96.2,99.7,90.2,90.9,4.0
step2 s3,100.0,99.6,95.4,95.4,99.7,89.2,103.0,1.2
wide45_lownoise s0,100.0,99.4,95.6,94.6,99.5,89.2,94.8,0.8
wide45_lownoise s2,100.0,99.3,96.4,94.8,99.1,90.7,89.8,0.8
wide45_lownoise s3,100.0,99.3,96.0,94.9,100.0,88.4,118.7,0.8
tcn_v2 s0,100.0,99.1,95.8,93.9,99.5,89.2,106.1,0.8
wide45_lownoise ens4,100.0,99.0,96.0,94.9,99.6,88.7,118.0,3.0


Độ lệch giữa các seed của cùng một họ (TB mọi góc):


,mean,min,max
step2,99.9,99.6,100.0
tcn_v2,98.2,97.6,99.1
wide45_lownoise,98.8,97.1,99.4


Đứng đầu bảng: step2 s0 (100.0). Trong vòng 2.0 điểm có 12 ứng viên: ['step2 s0', 'step2 s1', 'step2 s2', 'step2 ens4', 'step2 s3', 'wide45_lownoise s0', 'wide45_lownoise s2', 'wide45_lownoise s3', 'tcn_v2 s0', 'wide45_lownoise ens4', 'tcn_v2 s1', 'tcn_v2 ens4']
MODEL ĐƯỢC CHỌN (nhanh nhất trong nhóm đó): tcn_v2 s1


In [9]:
#cell 9: chọn ngưỡng tin cậy bằng VAL (ngưỡng nhỏ nhất mà độ chính xác khi tin cậy >= ngưỡng đạt ACC_TARGET) và ghi final_model_v2.json
ACC_TARGET = 95.0                                            # % (tham số tự đặt)
def thr_table(P, y, Mk):
    conf, pred = P.max(-1); ok = (pred == y[:, None]) & Mk; rows = []
    for t in THRS:
        sel = (conf >= t) & Mk; hit = ok & (conf >= t)
        rows.append({"ngưỡng": t,
                     "acc khi tin cậy>=ngưỡng": (ok & sel).sum().item() / max(sel.sum().item(), 1) * 100,
                     "độ phủ": sel.sum().item() / max(Mk.sum().item(), 1) * 100,
                     "% video đúng và đủ tin cậy": hit.any(1).float().mean().item() * 100})
    return pd.DataFrame(rows)
tv = thr_table(last_probs(CANDS[BEST], Xv, Mv), yv, Mv)
print(f"Ngưỡng tin cậy của {BEST} trên VAL:"); display(tv.round(1))
okt = tv[tv["acc khi tin cậy>=ngưỡng"] >= ACC_TARGET]
THR = float(okt["ngưỡng"].iloc[0]) if len(okt) else float(tv["ngưỡng"].iloc[-1])
print(f"NGƯỠNG ĐƯỢC CHỌN: {THR}" + ("" if len(okt) else f" (không ngưỡng nào đạt {ACC_TARGET}%, lấy ngưỡng cao nhất)"))
final = {"model": BEST, "checkpoints": [str(ck_path(*k)) for k in CANDS[BEST]], "n_in": [VARIANTS[k[0]]["n_in"] for k in CANDS[BEST]],
         "nguong": THR, "chon_tren": "val (manifest_v2)", "TB_moi_goc_val": float(tab.loc[BEST, "TB mọi góc"]),
         "quy_tac": f"xếp theo TB mọi góc trên val; chênh dưới {TOL} điểm thì lấy bản nhanh hơn; ngưỡng nhỏ nhất có acc >= {ACC_TARGET}% trên val"}
(CK / "final_model_v2.json").write_text(json.dumps(final, indent=2, ensure_ascii=False), encoding="utf-8")
print("Đã ghi", CK / "final_model_v2.json", "(trước khi nhìn test)")

Ngưỡng tin cậy của tcn_v2 s1 trên VAL:


,ngưỡng,acc khi tin cậy>=ngưỡng,độ phủ,% video đúng và đủ tin cậy
0,0.5,98.3,94.4,100.0
1,0.6,99.0,91.4,100.0
2,0.7,99.2,89.0,100.0
3,0.8,99.8,85.0,100.0
4,0.9,100.0,78.4,100.0


NGƯỠNG ĐƯỢC CHỌN: 0.5
Đã ghi C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9_v2\final_model_v2.json (trước khi nhìn test)


In [10]:
#cell 10: ĐÁNH GIÁ TEST MỘT LẦN cho model đã chọn. Chạy xong thì không chạy lại (muốn chạy lại phải xóa test_result_v2.json, và khi đó test không còn độc lập)
f = CK / "test_result_v2.json"
assert not f.exists(), f"Test đã chạy rồi ({f}). Không chạy lại để test vẫn độc lập."
out = {}
for sub, idx in SUB.items():
    P = last_probs(CANDS[BEST], Xte[idx], Mte[idx])
    r = metrics(P, yte[idx], Mte[idx]); tt = thr_table(P, yte[idx], Mte[idx]); row = tt[tt["ngưỡng"] == THR].iloc[0]
    out[sub] = {"n_video": len(idx), **r, "nguong": THR, "acc_khi_tin_cay": float(row["acc khi tin cậy>=ngưỡng"]),
                "do_phu": float(row["độ phủ"]), "pct_video_dung_du_tin_cay": float(row["% video đúng và đủ tin cậy"])}
f.write_text(json.dumps({"model": BEST, "ket_qua": out}, indent=2, ensure_ascii=False), encoding="utf-8")
show = pd.DataFrame(out).T[["n_video", "acc_all", "tb", "v50", "v60", "v70", "acc_khi_tin_cay", "do_phu", "pct_video_dung_du_tin_cay"]]
show.columns = ["số video", "acc mọi khung", "TB 5 ngưỡng", "% video @0.5", "@0.6", "@0.7", f"acc khi tin cậy>={THR}", "độ phủ", f"% video đúng và đủ tin cậy @{THR}"]
print(f"KẾT QUẢ TEST ({BEST}); all = 39 clip, nguoi_khac = person_01 (L3), sang = Sang (B1, B3):"); display(show.round(1))
print("Đã lưu:", f)

KẾT QUẢ TEST (tcn_v2 s1); all = 39 clip, nguoi_khac = person_01 (L3), sang = Sang (B1, B3):


,số video,acc mọi khung,TB 5 ngưỡng,% video @0.5,@0.6,@0.7,acc khi tin cậy>=0.5,độ phủ,% video đúng và đủ tin cậy @0.5
all,39.0,76.5,73.8,89.7,89.7,89.7,79.0,93.6,89.7
nguoi_khac,15.0,99.5,68.0,100.0,100.0,100.0,99.9,96.3,100.0
sang,24.0,61.9,77.5,83.3,83.3,83.3,65.1,91.8,83.3


Đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9_v2\test_result_v2.json


In [11]:
#cell 11: thí nghiệm người mới: train chỉ trên dữ liệu của Sang (Hand + Bonus), kiểm tra trên Sang (cảnh mới) và trên toàn bộ person_01 (người khác)
fam = BEST.rsplit(" ", 1)[0]                                 # họ của model đã chọn
sang_tr = np.where(meta["train"].nguon.ne("dataset_labeled"))[0]; sang_va = np.where(meta["val"].nguon.ne("dataset_labeled"))[0]
pick = lambda d, ix: tuple(t[torch.tensor(ix, device=dev)] for t in d)
tr_c, va_c = pick(data["train"], sang_tr), pick(data["val"], sang_va)
print(f"họ {fam} | train Sang: {len(sang_tr)} clip | val Sang (chọn epoch): {len(sang_va)} clip")
ckc = lambda s: CK / "nguoi_moi" / fam / f"seed{s}" / "TCN.pt"
for s in SEEDS: fit(fam, s, ckc(s), tr_c, va_c, "nguoi_moi")

dfl = m[m.nguon == "dataset_labeled"].reset_index(drop=True); Xl, yl, Ml = build(dfl)      # toàn bộ person_01
sets = {"Sang (test B1, B3: cảnh mới, cùng người)": (Xte[SUB["sang"]], yte[SUB["sang"]], Mte[SUB["sang"]]),
        "person_01 (người khác, toàn bộ 134 clip)": (Xl, yl, Ml)}
rows = []
for s in SEEDS:
    net = TCN(VARIANTS[fam]["n_in"]).to(dev); net.load_state_dict(torch.load(ckc(s), map_location=dev, weights_only=False)["w"]); net.eval()
    for nm, (X, y, Mk) in sets.items():
        with torch.no_grad(): P = net(VARIANTS[fam]["feat"](X, Mk)).softmax(-1)
        r = metrics(P, y, Mk)
        rows.append({"tập": nm, "seed": s, "acc mọi khung": r["acc_all"], "TB 5 ngưỡng": r["tb"], "acc khi tin cậy>=0.7": r["accc70"], "độ phủ 0.7": r["cov70"]})
rc = pd.DataFrame(rows); rc.to_csv(CK / "nguoi_moi_result.csv", index=False)
display(rc.groupby("tập")[["acc mọi khung", "TB 5 ngưỡng", "acc khi tin cậy>=0.7", "độ phủ 0.7"]].agg(["mean", "min", "max"]).round(1))
print("Đã lưu:", CK / "nguoi_moi_result.csv")

họ tcn_v2 | train Sang: 129 clip | val Sang (chọn epoch): 18 clip
nguoi_moi tcn_v2 seed 0: tốt nhất epoch 79 | dừng ở epoch 99 | val_loss 0.2353 | 4s
nguoi_moi tcn_v2 seed 1: tốt nhất epoch 25 | dừng ở epoch 45 | val_loss 0.2895 | 2s
nguoi_moi tcn_v2 seed 2: tốt nhất epoch 40 | dừng ở epoch 60 | val_loss 0.2713 | 2s
nguoi_moi tcn_v2 seed 3: tốt nhất epoch 42 | dừng ở epoch 62 | val_loss 0.2782 | 3s


acc mọi khung              \
                                                  mean   min   max   
tập                                                                  
Sang (test B1, B3: cảnh mới, cùng người)          60.4  57.1  64.2   
person_01 (người khác, toàn bộ 134 clip)          92.9  88.6  95.6   

                                         TB 5 ngưỡng              \
                                                mean   min   max   
tập                                                                
Sang (test B1, B3: cảnh mới, cùng người)        71.5  65.8  78.3   
person_01 (người khác, toàn bộ 134 clip)        94.2  91.9  96.3   

                                         acc khi tin cậy>=0.7              \
                                                         mean   min   max   
tập                                                                         
Sang (test B1, B3: cảnh mới, cùng người)                 78.2  73.4  88.4   
person_01 (người khác, toàn bộ 134 clip)                 94.5  89.7  97.4   

                                         độ phủ 0.7              
                                               mean   min   max  
tập                                                              
Sang (test B1, B3: cảnh mới, cùng người)       54.1  46.1  60.5  
person_01 (người khác, toàn bộ 134 clip)       84.5  82.1  88.7

Đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9_v2\nguoi_moi_result.csv
